# Frequency- and time-frequency-domain source-space analysis

MNE has several 

## Get epochs

In [ ]:
import numpy as np

import mne
from mne.beamformer import apply_dics_csd, make_dics
from mne.datasets import somato
from mne.time_frequency import csd_morlet

data_path = somato.data_path()
subject = "01"
task = "somato"
raw_fname = data_path / f"sub-{subject}" / "meg" / f"sub-{subject}_task-{task}_meg.fif"
# Use a shorter segment of raw just for speed here
raw = mne.io.read_raw_fif(raw_fname)
raw.crop(0, 120)  # one minute for speed
# Get epochs
events = mne.find_events(raw)
epochs = mne.Epochs(raw, events, event_id=1, tmin=-1.5, tmax=2, preload=True)

## Make DICS beamformer

In [ ]:
# Paths to forward operator and FreeSurfer subject directory
fname_fwd = (
    data_path / "derivatives" / f"sub-{subject}" / f"sub-{subject}_task-{task}-fwd.fif"
)
subjects_dir = data_path / "derivatives" / "freesurfer" / "subjects"
# Set frequencies
freqs = np.logspace(np.log10(12), np.log10(30), 10)
# Get cross-spectral density during baseline and trial periods
csd = csd_morlet(epochs, freqs, tmin=-1, tmax=1.5, decim=20)
csd_baseline = csd_morlet(epochs, freqs, tmin=-1, tmax=0, decim=20)
fwd = mne.read_forward_solution(fname_fwd)
dics = make_dics(
    epochs.info,
    fwd,
    csd,
    noise_csd=csd_baseline,
    pick_ori="max-power",
    reduce_rank=True,
    real_filter=True,
)

## Apply DICS beamformer

In [ ]:
stcs = []
labels = []
for epoch_idx in range(len(epochs)):
    epoch = epochs[epoch_idx]
    # Baseline CSD
    csd_baseline = csd_morlet(epoch, freqs, tmin=-1, tmax=0, decim=20)
    src_baseline, freqs = apply_dics_csd(csd_baseline, dics)
    stcs.append(src_baseline)
    labels.append('baseline')
    csd_trial = csd_morlet(epoch, freqs, tmin=0.5, tmax=1.5, decim=20)
    src_trial, freqs = apply_dics_csd(csd_trial, dics)
    stcs.append(src_trial)
    labels.append('trial')